In [3]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
model = OpenAIChatCompletionClient(model="gpt-4o-mini")

clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message="""You are an expert editor focused on clarity and simplicity. 
            Your job is to eliminate ambiguity, redundancy, and make every sentence crisp and clear. 
            Don't worry about persuasion or tone — just make the message easy to read and understand.""",
)

tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message="""You are a communication coach focused on emotional tone and professionalism. 
            Your job is to make the email sound warm, confident, and human — while staying professional 
            and appropriate for the audience. Improve the emotional resonance, polish the phrasing, 
            and adjust any words that may come off as stiff, cold, or overly casual.""",
)

persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message="""You are a persuasion expert trained in marketing, behavioral psychology, 
            and copywriting. Your job is to enhance the email's persuasive power: improve call to action, structure arguments, and emphasize benefits. Remove weak or passive language.""",
)

synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message="""You are an advanced email-writing specialist. Your role is to read all 
            prior agent responses and revisions, and then **synthesize the best ideas** into a unified, 
            polished draft of the email. Focus on: Integrating clarity, tone, and persuasion improvements; 
            Ensuring coherence, fluency, and a natural voice; Creating a version that feels professional, 
            effective, and readable.""",
)

critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message="""You are an email quality evaluator. Your job is to perform a final review 
            of the synthesized email and determine if it meets professional standards. Review the email for: 
            Clarity and flow, appropriate professional tone, effective call-to-action, and overall coherence.
            Be constructive but decisive. If the email has major flaws (unclear message, unprofessional tone, 
            or missing key elements), provide ONE specific improvement suggestion. If the email meets professional standards and communicates effectively, respond with 'The email meets professional standards.' followed by `TERMINATE` on a new line. You should only approve emails that are perfect enough for professional use, dont settle.""",
)

In [5]:
text_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=30)

termination_condition = text_termination | max_messages_termination

In [6]:
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent,
    ],
    termination_condition=termination_condition,
)

await Console(
  team.run_stream(task="Hi! I'm hungry, buy me lunch and invest in my business. Thanks.")
)

---------- TextMessage (user) ----------
Hi! I'm hungry, buy me lunch and invest in my business. Thanks.
---------- TextMessage (ClarityAgent) ----------
Hi! I’m hungry. Please buy me lunch and invest in my business. Thanks.
---------- TextMessage (ToneAgent) ----------
Subject: Lunch Invitation and Business Discussion

Hi [Recipient's Name],

I hope this message finds you well! I was wondering if you might be open to grabbing lunch together soon. I’d love the opportunity to share some ideas about my business and explore the potential for investment.

Thank you for considering it! I really appreciate your support.

Looking forward to hearing from you!

Best,  
[Your Name]  
---------- TextMessage (PersuasionAgent) ----------
Subject: Let's Discuss a Delicious Opportunity Over Lunch!

Hi [Recipient's Name],

I hope you’re doing well! I’d like to invite you to lunch soon—not just to satisfy my hunger, but to present an exciting opportunity that could benefit us both.

During our meal, I’

TaskResult(messages=[TextMessage(id='049893f4-2465-4fd5-a007-c955c567e780', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 10, 6, 8, 44, 661221, tzinfo=datetime.timezone.utc), content="Hi! I'm hungry, buy me lunch and invest in my business. Thanks.", type='TextMessage'), TextMessage(id='69cd3e99-9f34-4405-978b-1da57c869730', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=77, completion_tokens=18), metadata={}, created_at=datetime.datetime(2026, 6, 10, 6, 8, 47, 106884, tzinfo=datetime.timezone.utc), content='Hi! I’m hungry. Please buy me lunch and invest in my business. Thanks.', type='TextMessage'), TextMessage(id='e892bf17-8051-41fc-b241-7410554b5fe3', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=123, completion_tokens=82), metadata={}, created_at=datetime.datetime(2026, 6, 10, 6, 8, 50, 11881, tzinfo=datetime.timezone.utc), content="Subject: Lunch Invitation and Business Discussion\n\nHi [Recipient's Name],\n\nI ho